<a href="https://colab.research.google.com/github/Imran1hp/YOLO-object-detection-model/blob/main/YOLOv8_Object_setection_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Download the Dataset from google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
file_path ="/content/drive/MyDrive/Waste_dataset_for_Yolov8"

In [3]:
import os
os.listdir(file_path)

['Waste.yolov8.zip', 'train', 'data.yaml', 'README.roboflow.txt']

In [4]:
import zipfile

zip_file_path = os.path.join(file_path, 'Waste.yolov8.zip')

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(file_path)
print(f"Unzipped {zip_file_path} to {file_path}")

Unzipped /content/drive/MyDrive/Waste_dataset_for_Yolov8/Waste.yolov8.zip to /content/drive/MyDrive/Waste_dataset_for_Yolov8


In [5]:
os.listdir(file_path)

['Waste.yolov8.zip', 'train', 'data.yaml', 'README.roboflow.txt']

In [7]:
data_path =  os.path.join(file_path , 'train')
os.listdir(data_path)

['images', 'labels']

In [8]:
data_path

'/content/drive/MyDrive/Waste_dataset_for_Yolov8/train'

# Split and creat the valid and test files

In [9]:
import os
import random
import shutil


image_path = data_path +"/images"
label_path = data_path +"/labels"

os.makedirs(file_path + "/valid/images" , exist_ok = True)
os.makedirs(file_path + "/valid/labels" , exist_ok = True)

os.makedirs(file_path + "/test/images" ,exist_ok= True)
os.makedirs(file_path + "/test/labels" , exist_ok = True)

images = os.listdir(image_path)
random.shuffle(images)


train_ratio = 0.7
valid_ratio = 0.2
test_ratio = 0.1
total = len(images)

train_end = int(total * train_ratio)
valid_end = train_end + int(total * valid_ratio)

train_files = images[:train_end]
valid_files = images[train_end:valid_end]
test_files = images[valid_end:]


In [10]:
def move_files(file_list , target_folder):
  for img_file in file_list:

    label_file = os.path.splitext(img_file)[0]+'.txt'

    shutil.move(
        os.path.join(image_path , img_file),
        os.path.join(file_path ,target_folder , 'images',img_file)
    )

    shutil.move(
        os.path.join(label_path , label_file),
        os.path.join(file_path ,  target_folder ,"labels" ,label_file)
    )


#filter the files with proper labels

In [11]:
import os

def filter_files_with_existing_labels(file_list, image_base_path, label_base_path):
    counter_for_skip_files =0;
    filtered_list = []
    for img_file in file_list:
        img_full_path = os.path.join(image_base_path, img_file)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_full_path = os.path.join(label_base_path, label_file)
        if os.path.exists(img_full_path) and os.path.exists(label_full_path):
            filtered_list.append(img_file)
        else:
            counter_for_skip_files = counter_for_skip_files+1
            print(f"Skipping {img_file} as its image or label file is missing.")

    print(f'Total number of images without proper label is {counter_for_skip_files}')
    return filtered_list



filtered_valid_files = filter_files_with_existing_labels(valid_files, image_path, label_path)
filtered_test_files = filter_files_with_existing_labels(test_files, image_path, label_path)


move_files(filtered_valid_files ,"valid" )
move_files(filtered_test_files , "test")

Total number of images without proper label is 0
Total number of images without proper label is 0


In [12]:
data_path

'/content/drive/MyDrive/Waste_dataset_for_Yolov8/train'

In [14]:
!ls /content/drive/MyDrive/Waste_dataset_for_Yolov8/valid/images | wc -l

504


In [15]:
!ls /content/drive/MyDrive/Waste_dataset_for_Yolov8/valid/labels | wc -l

504


#Install the YOlo model from Ultralytics

In [17]:
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.0 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO


In [ ]:
import yaml
import os

with open(os.path.join(file_path, 'data.yaml'), 'r') as f:
    data = yaml.safe_load(f)
print(data)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 6, 'names': ['Metal', 'Paper', 'Plastic', 'Random Trash', 'cardboard', 'glass'], 'roboflow': {'workspace': 'imran-laskar', 'project': 'imran-laskar', 'version': 'dataset', 'license': 'Private', 'url': 'https://app.roboflow.com/imran-laskar/imran-laskar/dataset'}}


In [ ]:
model = YOLO('yolov8n.pt')

In [ ]:
model.train(data=os.path.join(file_path, 'data.yaml'), epochs=10, imgsz=640, patience=50)